In [31]:
import pandas as pd
import numpy as np
url="https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)



In [32]:
len(df)
df.dtypes

model_year               int64
origin                  object
fuel_type               object
drivetrain              object
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [33]:
base=['engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg']

In [34]:
df[base].isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [35]:
# qustion 1
col=['engine_displacement'
,'horsepower'
,'vehicle_weight'
,'model_year']
df[col].isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

In [36]:
df=df[base].fillna(0)
y=df['fuel_efficiency_mpg'].values

In [37]:
n=len(df)
n_val=int(n*0.2)
n_test=int(n*0.2)
n_train=n-n_test-n_val
n,n_val+n_test+n_train

(10000, 10000)

In [38]:
idx=np.arange(n)
np.random.seed(42)
np.random.shuffle(idx)
df_train=df.iloc[idx[:n_train]]
df_val=df.iloc[idx[n_train:n_val+n_train]]
df_test=df.iloc[idx[n_val+n_train:]]


In [39]:
len(df_train),len(df_val),len(df_test)

(6000, 2000, 2000)

In [40]:
# question 2
hp_median = df['horsepower'].median()
print(f"Median of Horsepower: {hp_median}")

Median of Horsepower: 252.0


In [41]:
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [46]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    # Normal Equation: w = (X^T * X)^(-1) * X^T * y
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [47]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    # Add Regularization term (r * I) to the diagonal
    reg = r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX + reg)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [48]:
# Split prepared dataset using seed 42
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

# Extract target variable y
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Features list
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

In [51]:
# Fill NAs with 0
X_train_zero = df_train[features].fillna(0).values
w0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

X_val_zero = df_val[features].fillna(0).values
y_pred_zero = w0_zero + X_val_zero.dot(w_zero)
score_zero = round(rmse(y_val, y_pred_zero), 3)

print(f"rsme with 0 imputation: {score_zero}")

rsme with 0 imputation: 2.205


In [52]:
# Calculate mean from train set ONLY
mean_ = df_train['horsepower'].mean()

# Fill null with mean
X_train_mean = df_train[features].fillna(mean_).values
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

X_val_mean = df_val[features].fillna(mean_).values
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
score_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"rsme with Mean imputation: {score_mean}")

rsme with Mean imputation: 2.205


In [54]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

X_train_zero = df_train[features].fillna(0).values
X_val_zero = df_val[features].fillna(0).values

for r in r_values:
    w0, w = train_linear_regression_reg(X_train_zero, y_train, r=r)
    y_pred = w0 + X_val_zero.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:6} | esme = {score}")

r =      0 | esme = 2.2053
r =   0.01 | esme = 2.2058
r =    0.1 | esme = 2.2241
r =      1 | esme = 2.3492
r =      5 | esme = 2.4094
r =     10 | esme = 2.4195
r =    100 | esme = 2.4292


In [55]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for seed in seeds:
    # 1. Shuffle
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    # 2. Split
    df_tr = df.iloc[idx[:n_train]].copy()
    df_v = df.iloc[idx[n_train:n_train + n_val]].copy()
    
    y_tr = df_tr['fuel_efficiency_mpg'].values
    y_v = df_v['fuel_efficiency_mpg'].values
    
    # 3. Fill NA with 0
    X_tr = df_tr[features].fillna(0).values
    X_v = df_v[features].fillna(0).values
    
    # 4. Train & Evaluate
    w0, w = train_linear_regression(X_tr, y_tr)
    y_pred = w0 + X_v.dot(w)
    score = rmse(y_v, y_pred)
    scores.append(score)

# Calculate Std Dev
std_dev = round(np.std(scores), 3)
print(f"Standard Deviation of scores: {std_dev}")

Standard Deviation of scores: 0.029


In [57]:
# 1. Split using seed 9
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

# 2. Combine Train and Validation
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# 3. Prepare Feature Matrices (Impute NAs with 0)
X_full_train = df_full_train[features].fillna(0).values
X_test = df_test[features].fillna(0).values

# 4. Train model with r = 0.001
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# 5. Predict and Evaluate on Test Set
y_pred_test = w0 + X_test.dot(w)
test_rmse = round(rmse(y_test, y_pred_test), 3)

print(f"Final Test rmse: {test_rmse}")

Final Test rmse: 2.236


In [60]:
import pandas as pd

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

median_correct = df['horsepower'].median()
print(f"Correct Median (without fillna): {median_correct}")

Correct Median (without fillna): 254.0
